# Akan-BPE Phase 2A1 — Qwen3-0.6B × Akan TTS Tokenizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/2a1_qwen3-0.6b_tts.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/2a1_qwen3-0.6b_tts.ipynb)


Self-contained Colab notebook. Run all cells top-to-bottom.

**Steps:**
1. Clone repo and install deps
2. Download Akan datasets from HuggingFace Hub
3. Run QLoRA fine-tune experiment (`Qwen/Qwen3-0.6B` + Akan TTS tokenizer, random embedding init)
4. Inspect results — fertility reduction, **bits-per-byte (BPB, tokenizer-agnostic)**, eval loss/perplexity, generation samples
5. **Embedding-init ablation (M2)** — re-run with mean-of-subword init and compare the two arms (BPB / perplexity)

**GPU required.** Before running: Runtime → Change runtime type → T4 GPU.

> The notebook runs **two** full QLoRA fine-tunes (random + mean_subword arms), ~45–50 min each on a T4.

In [1]:
# 1. Clone repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

# Guard against triple-nesting on cell re-run: only clone+cd when we are not
# already sitting inside the repo root.
if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

Cloning into 'akan-bpe'...
remote: Enumerating objects: 464, done.
remote: Counting objects: 100% (464/464), done.
remote: Compressing objects: 100% (297/297), done.
remote: Total 464 (delta 280), reused 330 (delta 152), pack-reused 0 (from 0)
Receiving objects: 100% (464/464), 918.34 KiB | 17.01 MiB/s, done.
Resolving deltas: 100% (280/280), done.
/kaggle/working/akan-bpe
Working directory: /kaggle/working/akan-bpe


In [2]:
# 2. Install dependencies
%pip install -q -e ".[dev,train]"
%pip install -q bitsandbytes sentencepiece

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.1/513.1 kB 31.2 MB/s eta 0:00:00
  Building editable for akan-bpe (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have

In [3]:
# 3. Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [4]:
# 4. Download Akan datasets from HuggingFace Hub
# Produces: data/pristine_twi_{train,validation,test}.jsonl
#           data/aka_asr_{train,validation,test}.jsonl
# --tts-limit 50000 pins the pristine-Twi corpus to the same 45,000/2,500/2,500
# split used in the report (the default would pull ~188k rows and shift the split).
!python scripts/download.py --output-dir data --tts-limit 50000

README.md: 29.2kB [00:00, 44.6MB/s]
Resolving data files: 100%|████████████████| 270/270 [00:00<00:00, 30134.70it/s]
Wrote 8085 rows to data/aka_asr_train.jsonl
Wrote 1011 rows to data/aka_asr_validation.jsonl
Wrote 1011 rows to data/aka_asr_test.jsonl
README.md: 4.06kB [00:00, 1.55MB/s]
Wrote 40000 rows to data/pristine_twi_train.jsonl
Wrote 5000 rows to data/pristine_twi_validation.jsonl
Wrote 5000 rows to data/pristine_twi_test.jsonl
Manifest written to data/download_manifest.json


In [5]:
# 5. Train TTS tokenizer (if not already present)
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    print("TTS tokenizer not found — training now ...")
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    sz = Path("models/tts_tokenizer.json").stat().st_size / 1e6
    print(f"TTS tokenizer already present ({sz:.1f} MB), skipping.")

TTS tokenizer already present (0.5 MB), skipping.


In [6]:
# 5. Verify all required inputs exist
from pathlib import Path

required = {
    "TTS train data" : Path("data/pristine_twi_train.jsonl"),
    "TTS test data"  : Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer"  : Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")
for name, p in required.items():
    print(f"  {name}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")

All inputs ready.
  TTS train data: data/pristine_twi_train.jsonl  (64.0 MB)
  TTS test data: data/pristine_twi_test.jsonl  (8.0 MB)
  TTS tokenizer: models/tts_tokenizer.json  (0.5 MB)


In [7]:
# 6. Run Phase 2A1 experiment — arm A: random embedding init (default)
# QLoRA fine-tune on Qwen3-0.6B with the Akan TTS tokenizer.
# Writes model adapters to models/phase2a1_qwen3_0_6b_tts/
# Writes result JSON to results/phase2a1_qwen3_0_6b_tts.json
# --embedding-init-mode random is the default; stated explicitly so this run pairs
# cleanly with the mean_subword ablation at the end of the notebook.
!python scripts/model_integration.py \
    --experiment-id phase2a1_qwen3_0_6b_tts \
    --model-id Qwen/Qwen3-0.6B-Base \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a1_qwen3_0_6b_tts \
    --results-output results/phase2a1_qwen3_0_6b_tts.json \
    --device-mode colab-qlora \
    --embedding-init-mode random \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.60MB/s]
tokenizer_config.json: 9.68kB [00:00, 20.2MB/s]
vocab.json: 2.78MB [00:00, 96.1MB/s]
merges.txt: 1.67MB [00:00, 115MB/s]
tokenizer.json: 7.03MB [00:00, 134MB/s]
model.safetensors: 100%|████████████████████| 1.19G/1.19G [00:05<00:00, 228MB/s]
Loading weights: 100%|█| 310/310 [00:00<00:00, 457.63it/s, Materializing param=m
generation_config.json: 100%|███████████████████| 138/138 [00:00<00:00, 624kB/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '7.112', 'grad_norm': '3.729', 'learning_rate': '0.000193', 'epoch': '

In [8]:
# 7. Load result JSON
import json
from pathlib import Path

result = json.loads(
    Path("results/phase2a1_qwen3_0_6b_tts.json").read_text(encoding="utf-8")
)
print("Experiment :", result["experiment_id"])
print("Model      :", result["model_id"])
print("Device     :", result.get("device", {}))

Experiment : phase2a1_qwen3_0_6b_tts
Model      : Qwen/Qwen3-0.6B-Base
Device     : {'cuda_available': True, 'device_count': 1, 'device_name': 'Tesla T4'}


In [9]:
# 8. Token count comparison — fertility reduction
cmp = result["token_count_comparison"]
print(f"Base tokenizer fertility : {cmp['base_model_tokenizer']['fertility']:.3f} tokens/word")
print(f"Akan tokenizer fertility : {cmp['experiment_tokenizer']['fertility']:.3f} tokens/word")
print(f"Reduction ratio          : {cmp['token_reduction_ratio']:.1%}")

Base tokenizer fertility : 2.530 tokens/word
Akan tokenizer fertility : 1.259 tokens/word
Reduction ratio          : 50.3%


In [10]:
# 9. Eval metrics
ev = result["eval"]
print(f"Eval loss  : {ev['eval_loss']:.4f}")
print(f"Perplexity : {ev['perplexity']:.2f}")

Eval loss  : 4.4146
Perplexity : 82.65


In [11]:
# 9b. Bits-per-byte (BPB) — tokenizer-agnostic cross-tokenizer comparison
# Perplexity is NOT comparable across tokenizers (different vocab → different token
# counts for the same text). BPB normalizes the model's total NLL by the fixed
# UTF-8 byte count of the eval text, so the base model and the Akan-tokenizer model
# are directly comparable. Lower BPB is better; positive improvement = Akan wins.
bpb = result["eval"]["bpb"]
exp = bpb["experiment"]
base = bpb["base"]

print(f"Embedding init     : {result['embedding_init_mode']}")
print(f"Eval bytes scored  : {bpb['total_bytes']:,}")
print(f"Akan TTS model BPB : {exp['bits_per_byte']:.4f} bits/byte")
if base is not None:
    print(f"Base model BPB     : {base['bits_per_byte']:.4f} bits/byte")
    print(f"Improvement        : {bpb['improvement']:+.4f} bits/byte (base − experiment; + = Akan better)")
else:
    print("Base model BPB     : (skipped — re-run without --skip-base-bpb to populate)")

Embedding init     : random
Eval bytes scored  : 769,514
Akan TTS model BPB : 1.0785 bits/byte
Base model BPB     : 1.1008 bits/byte
Improvement        : +0.0222 bits/byte (base − experiment; + = Akan better)


In [12]:
# 10. Generation samples
for i, s in enumerate(result["generation_samples"], 1):
    print(f"--- Sample {i} ---")
    print(f"Prompt    : {s['prompt']}")
    print(f"Completion: {s['completion']}")
    print()

--- Sample 1 ---
Prompt    : Ananse nyaa adwene sɛ ɔbɛkɔ akɔkyerɛ agyapadie a ɛwɔ abɔdeɛ mu. Ɔkɔɔ
Completion: Ananse nyaa adwene sɛ ɔbɛkɔ akɔkyerɛ agyapadie a ɛwɔ abɔdeɛ mu . Ɔkɔɔ nea ɛrekɔ so wɔ yɛn man Ghana mu . Ananse kɔɔ kwaeɛ a , ɛyɛ me nwanwa sɛ me ne me ho . Ananse kɔɔ kwaeɛ a , me ne me ho

--- Sample 2 ---
Prompt    : Ɛnnɛ me tirim yɛ me duru pa ara bere a mifi akwantu
Completion: Ɛnnɛ me tirim yɛ me duru pa ara bere a mi fi akwantu a , me werɛ how paa . Sɛ mehwɛ sɛnea me dwene me sɛ me dwene me ho a , me dwene me ho . Sɛ mehwɛ sɛnea me dwene me sɛ

--- Sample 3 ---
Prompt    : Sɛ yɛbɛka abakɔsɛm a, nnansa yi, anigyeɛ kɛse bi a ɛbaa Prisons
Completion: Sɛ yɛbɛka abakɔsɛm a , nnansa yi , anigyeɛ kɛse bi a ɛbaa Prisons , na ɛyɛ me nwanwa . Sɛ wose " Me " a , me ne me adwene a , me ne me adwene a , me ne me adwene a , me



## Embedding-init ablation (M2) — random vs mean-of-subword

The run above initialized the new Akan-vocab embedding rows with the default **random**
scheme. M2 adds a modeling alternative: `--embedding-init-mode mean_subword` seeds each
Akan token's embedding from the **mean of the base model's subword embeddings** for that
token's surface string (falling back to the global embedding mean for tokens with no base
subwords). The same scheme is applied to the LM head when output embeddings are untied.

The cells below run the `mean_subword` arm as a clean A/B — identical data, tokenizer, and
hyperparameters, only the init differs — then compare the two arms on BPB and perplexity.

> **This is a second full QLoRA fine-tune (~45–50 min on a T4).** It writes a separate result
> JSON (`results/phase2a1_qwen3_0_6b_tts_meansub.json`) and adapter dir, so the random-arm
> artifacts above are left untouched.

In [13]:
# 11. Ablation arm B: mean-of-subword embedding init (second full QLoRA run)
!python scripts/model_integration.py \
    --experiment-id phase2a1_qwen3_0_6b_tts_meansub \
    --model-id Qwen/Qwen3-0.6B-Base \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a1_qwen3_0_6b_tts_meansub \
    --results-output results/phase2a1_qwen3_0_6b_tts_meansub.json \
    --device-mode colab-qlora \
    --embedding-init-mode mean_subword \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

Loading weights: 100%|█| 310/310 [00:00<00:00, 514.71it/s, Materializing param=m
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
{'loss': '6.711', 'grad_norm': '3.905', 'learning_rate': '0.000193', 'epoch': '0.03906'}
{'loss': '5.587', 'grad_norm': '2.102', 'learning_rate': '0.0001852', 'epoch': '0.07812'}
{'loss': '5.305', 'grad_norm': '2.407', 'learning_rate': '0.0001773', 'epoch': '0.1172'}
{'loss': '5.066', 'grad_norm': '2.942', 'learning_rate': '0.0001695', 'epoch': '0.1562'}
{'loss': '4.841', 'grad_norm': '2.56', 'learning_rate': '0.0001617', 'epoch': '0.1953'}
{'loss': '4.651', 'grad_norm': '2.623'

In [14]:
# 12. Compare embedding-init arms: random (arm A) vs mean_subword (arm B)
import json
from pathlib import Path


def load_eval(path):
    r = json.loads(Path(path).read_text(encoding="utf-8"))
    b = r["eval"]["bpb"]
    return {
        "init": r["embedding_init_mode"],
        "rows_initialized": (r.get("embedding_init") or {}).get("rows_initialized"),
        "eval_loss": r["eval"]["eval_loss"],
        "perplexity": r["eval"]["perplexity"],
        "exp_bpb": b["experiment"]["bits_per_byte"],
        "base_bpb": (b["base"] or {}).get("bits_per_byte"),
        "improvement": b.get("improvement"),
    }


arms = {
    "random": load_eval("results/phase2a1_qwen3_0_6b_tts.json"),
    "mean_subword": load_eval("results/phase2a1_qwen3_0_6b_tts_meansub.json"),
}

header = f"{'metric':<24}{'random':>14}{'mean_subword':>16}"
print(header)
print("-" * len(header))


def show(label, key, fmt="{:.4f}"):
    cells = []
    for arm in ("random", "mean_subword"):
        v = arms[arm][key]
        cells.append(fmt.format(v) if v is not None else "—")
    print(f"{label:<24}{cells[0]:>14}{cells[1]:>16}")


show("eval_loss", "eval_loss")
show("perplexity", "perplexity", "{:.2f}")
show("experiment BPB", "exp_bpb")
show("base BPB", "base_bpb")
show("improvement (base−exp)", "improvement")

best = min(arms, key=lambda a: arms[a]["exp_bpb"])
print(f"\nLower fine-tuned BPB: {best!r} arm.")

metric                          random    mean_subword
------------------------------------------------------
eval_loss                       4.4146          3.8144
perplexity                       82.65           45.35
experiment BPB                  1.0785          0.9319
base BPB                        1.1008          1.1008
improvement (base−exp)          0.0222          0.1688

Lower fine-tuned BPB: 'mean_subword' arm.
